# Epic on FHIR Auth Model

Notebook for Epic on FHIR auth model workflows.

In [ ]:
# Initialize Databricks Connect when running locally (spark/dbutils are pre-injected on Databricks)
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().profile("fe-vm-mkgs-databricks-demos").getOrCreate()

In [ ]:
# Widget definitions (defaults from bundle variables)
# catalog_use, schema_use, secret_scope_name, client_id_dbs_key, algo, token_url,
# mlflow_experiment_name (deployed experiment), registered_model_name (deployed model)

_catalog_use = "mkgs_dev"
_schema_use = "dev_matthew_giglia_epic_on_fhir"
_secret_scope_name = "epic_on_fhir_oauth_keys"
_client_id_dbs_key = "client_id"
_algo = "RS384"
_token_url = "https://fhir.epic.com/interconnect-fhir-oauth/oauth2/token"
_mlflow_experiment_name = "/Workspace/experiments/[dev matthew_giglia] epic_on_fhir_auth"
_registered_model_name = "mkgs_dev.dev_matthew_giglia_epic_on_fhir.epic_on_fhir_auth"

try:
    dbutils.widgets.text("catalog_use", _catalog_use, "Catalog")
    dbutils.widgets.text("schema_use", _schema_use, "Schema")
    dbutils.widgets.text("secret_scope_name", _secret_scope_name, "Epic Secrets Scope")
    dbutils.widgets.text("client_id_dbs_key", _client_id_dbs_key, "Epic Client ID DB Secrets Key")
    dbutils.widgets.text("algo", _algo, "Epic Token Encryption Algorithm")
    dbutils.widgets.text("token_url", _token_url, "Epic Token URL")
    dbutils.widgets.text("mlflow_experiment_name", _mlflow_experiment_name, "MLflow Experiment Name")
    dbutils.widgets.text("registered_model_name", _registered_model_name, "Registered Model Name")
except Exception:
    pass  # Databricks Connect: widget creation may not be fully supported

In [ ]:
# Retrieve widget values (fallback for Databricks Connect)
try:
    catalog_use = dbutils.widgets.get("catalog_use")
    schema_use = dbutils.widgets.get("schema_use")
    secret_scope_name = dbutils.widgets.get("secret_scope_name")
    client_id_dbs_key = dbutils.widgets.get("client_id_dbs_key")
    algo = dbutils.widgets.get("algo")
    token_url = dbutils.widgets.get("token_url")
    mlflow_experiment_name = dbutils.widgets.get("mlflow_experiment_name")
    registered_model_name = dbutils.widgets.get("registered_model_name")
except (AttributeError, Exception):
    catalog_use = _catalog_use
    schema_use = _schema_use
    secret_scope_name = _secret_scope_name
    client_id_dbs_key = _client_id_dbs_key
    algo = _algo
    token_url = _token_url
    mlflow_experiment_name = _mlflow_experiment_name
    registered_model_name = _registered_model_name

In [ ]:
import sys
from pathlib import Path
# Ensure epic_on_fhir package is importable
_root = Path().resolve()
if _root.name == "src":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pandas as pd
import mlflow
from epic_on_fhir.epic_fhir_pyfunc import EpicFhirPyfuncModel

# Use experiment and registered model from widget defaults
mlflow.set_experiment(mlflow_experiment_name)

In [ ]:
# Build the pyfunc model with widget values
model = EpicFhirPyfuncModel(
    secret_scope_name=secret_scope_name,
    client_id_dbs_key=client_id_dbs_key,
    token_url=token_url,
    algo=algo,
)

In [ ]:
import json
from pathlib import Path

In [ ]:
# Input example for signature: http_method, resource, action, data (optional)
# Row 1: Patient GET | Row 2: Observation create | Row 3: AllergyIntolerance create

import random

# Keep date 2019-09-05, randomly select a valid time for effectiveDateTime
_obs_date = "2019-09-05"
_obs_time = f"{random.randint(0, 23):02d}:{random.randint(0, 59):02d}:{random.randint(0, 59):02d}"
_effective_dt = f"{_obs_date}T{_obs_time}Z"

observation_payload = {
    "resourceType": "Observation",
    "status": "final",
    "category": [{"coding": [{"system": "http://hl7.org/fhir/observation-category", "code": "vital-signs", "display": "Vital Signs"}], "text": "Vital Signs"}],
    "code": {"coding": [{"system": "urn:oid:1.2.840.114350.1.13.0.1.7.2.707679", "code": "8", "display": "Heart Rate"}], "text": "Heart Rate"},
    "subject": {"reference": "Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
    "encounter": {"reference": "Encounter/e0u1fd.jUCNqz8ZQuTaMtsQ3"},
    "effectiveDateTime": _effective_dt,
    "valueQuantity": {"value": 75},
}

allergy_payload = {
    "resourceType": "AllergyIntolerance",
    "clinicalStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/allergyintolerance-clinical", "code": "active", "display": "Active"}], "text": "Active"},
    "verificationStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/allergyintolerance-verification", "code": "unconfirmed", "display": "Unconfirmed"}], "text": "Unconfirmed"},
    "type": "allergy",
    "category": ["medication"],
    "criticality": "low",
    "code": {"coding": [{"system": "http://www.nlm.nih.gov/research/umls/rxnorm", "code": "7980", "display": "Penicillin G"}], "text": "Penicillin"},
    "patient": {"reference": "Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
    "recorder": {"reference": "Practitioner/eM5CWtq15N0WJeuCet5bJlQ3", "display": "Physician Family Medicine, MD"},
    "recordedDate": "2024-01-15",
    "reaction": [{"manifestation": [{"coding": [{"system": "http://snomed.info/sct", "code": "247472004", "display": "Hives"}], "text": "Hives"}]}],
}

input_example = pd.DataFrame([
    {"http_method": "get", "resource": "Patient", "action": "T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
    {"http_method": "post", "resource": "Observation", "action": "", "data": json.dumps(observation_payload)},
    {"http_method": "post", "resource": "AllergyIntolerance", "action": "", "data": json.dumps(allergy_payload)},
])

In [ ]:
# Conda env for model serving: JWT (PyJWT), cryptography (for RS384), requests, pandas, mlflow
_conda_env = {
    "name": "epic_on_fhir_serving",
    "channels": ["conda-forge"],
    "dependencies": [
        "python=3.12",
        "pip",
        {"pip": ["PyJWT", "cryptography", "requests", "pandas", "mlflow>=3.1"]},
    ],
}

In [ ]:
# Include epic_on_fhir package source for serving (not pip-installable in serving env)
_src_path = Path(_root) / "src"
_code_path = [str(_src_path)] if _src_path.exists() else []

# Infer MLflow signature from model_input (optionally run model.predict(input_example) for output schema)
from mlflow.models import infer_signature

try:
    _model_output = model.predict(input_example)
    _signature = infer_signature(model_input=input_example, model_output=_model_output)
except Exception:
    _signature = infer_signature(model_input=input_example)

In [ ]:

# Log model to MLflow 3 and register (no start_run needed; models are first-class entities)
model_info = mlflow.pyfunc.log_model(
    name="model",
    python_model=model,
    registered_model_name=registered_model_name,
    input_example=input_example,
    signature=_signature,
    conda_env=_conda_env,
    code_path=_code_path if _code_path else None,
    params={
        "secret_scope_name": secret_scope_name,
        "client_id_dbs_key": client_id_dbs_key,
        "token_url": token_url,
        "algo": algo,
    },
)
model_info  # display model_uri, model_id

In [ ]:
# Optional: test prediction (requires Epic secrets in scope). Uncomment to run.
# In MLflow 3, use model_uri from log_model result or models:/name/version
# Same example as input_example: http_method, resource, action
# test_df = pd.DataFrame([{"http_method": "get", "resource": "Patient", "action": "T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"}])
# loaded = mlflow.pyfunc.load_model(model_info.model_uri)  # from cell above
# # or: mlflow.pyfunc.load_model(f"models:/{registered_model_name}/1")
# loaded.predict(test_df)